# Stage 1 — SEC Cash-Tax Data Construction & QA

**Executed coding-sample notebook · review copy**

This notebook extracts SEC cash-tax facts, reviews duplicate representations and reporting scopes, and records unresolved accounting exceptions.

- Original 10-K filing cohort: **2,340**
- Current annual dimensionless USD Total coverage: **1,159**
- Provisionally eligible filing-level Totals: **1,157**
- Complete four-category Strict scopes: **184**; exact matches: **177**; registered exceptions: **7**
- Stage 1 status: **CONDITIONAL PASS**

The accounting logic and saved analysis outputs are preserved. Repository-relative paths and this navigation page have been adapted for this project layout. See [`docs/reproduction.md`](../docs/reproduction.md) before rerunning. The raw SEC input is not included in this repository.

# SEC Cash Tax Data Construction and Quality Assurance
Stage 1 coding sample · March 2026 pilot · rule freeze: 2026-11-11.

This notebook constructs cash income tax facts, tests duplicates and reconciliation,
and audits filing-level Total eligibility. It does not estimate regressions or build
a geographic research extension. Reported facts remain separate from diagnostic
calculations and provisional analytical observations.

Run all cells in order from a fresh Python kernel. See `README.md` and `docs/reproduction.md` for setup.
The development `.ipynb`, not its HTML export, is the code source. Repeated previews
and superseded exploratory classifications are omitted; the final classifications
and the seven original reconciliation exceptions are retained.


## 1. Configuration and immutable inputs
Set `SEC_NOTES_ZIP` to a downloaded public SEC March 2026 Notes ZIP, or place it at
`data/raw/2026_03_notes.zip`. A relative sibling-project path is also supported.
Outputs are written only under `outputs/stage1/`.
The hash pins this pilot snapshot; a different release requires renewed validation.


In [1]:
from pathlib import Path
from zipfile import ZipFile
from decimal import Decimal
import os, sys, json, hashlib
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, HERE.parent) if (p / 'src' / 'tax_data_utils.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Launch Jupyter from the repository root or notebooks/; see README.md.')
sys.path.insert(0, str(ROOT / 'src'))
from tax_data_utils import audit_duplicates, build_pilot_outputs, audit_total_eligibility
configured = os.environ.get('SEC_NOTES_ZIP')
candidates = [Path(configured)] if configured else [ROOT / 'data/raw/2026_03_notes.zip']
notes_path = next((p.resolve() for p in candidates if p.is_file()), None)
if notes_path is None:
    raise FileNotFoundError('Missing 2026_03_notes.zip. Set SEC_NOTES_ZIP or see README.')
raw_sha256 = hashlib.file_digest(notes_path.open('rb'), 'sha256').hexdigest()
assert raw_sha256 == 'd1c1711e6cee2f197fb91879521da178329d1c1ed369f9086aa501aff45f83d4', 'Different source release: revalidate pilot.'
OUTPUT = ROOT / 'outputs' / 'stage1'
OUTPUT.mkdir(parents=True, exist_ok=True)
fact_key = ['adsh','tag','version','ddate','qtrs','uom','dimh']
with ZipFile(notes_path) as zf:
    assert {'sub.tsv','num.tsv','tag.tsv','dim.tsv','cal.tsv'}.issubset(zf.namelist())
    sub = pd.read_csv(zf.open('sub.tsv'), sep='\t', dtype='string')
    dim = pd.read_csv(zf.open('dim.tsv'), sep='\t', dtype='string')
    tag_meta = pd.read_csv(zf.open('tag.tsv'), sep='\t', dtype='string')
assert sub['adsh'].is_unique and dim['dimhash'].is_unique
print('Input verified:', notes_path.name, raw_sha256)
print('SUB:', len(sub), 'DIM:', len(dim))


Input verified: 2026_03_notes.zip d1c1711e6cee2f197fb91879521da178329d1c1ed369f9086aa501aff45f83d4
SUB: 9679 DIM: 357052


## 2. Concept discovery and raw NUM extraction
Four standard concepts define Total/Federal/State-Local/Foreign. Filing extension
concepts containing `IncomeTaxPaidOther` remain separate. Taxonomy versions and all
16 NUM fields are preserved; no amount is imputed or converted to another currency.


In [2]:
# Core cash-tax tags
core_tax_tags = [
    "IncomeTaxesPaidNet",
    "IncomeTaxPaidFederalAfterRefundReceived",
    "IncomeTaxPaidStateAndLocalAfterRefundReceived",
    "IncomeTaxPaidForeignAfterRefundReceived"
]

# Discover all custom Other cash-tax tags
other_tags = sorted(
    tag_meta.loc[
        tag_meta["tag"].str.contains(
            "IncomeTaxPaidOther",
            regex=False,
            na=False
        ),
        "tag"
    ].dropna().unique().tolist()
)

# Full extraction list
tax_tags = list(dict.fromkeys(core_tax_tags + other_tags))

print("Core tags:", len(core_tax_tags))
print("Other tags:", len(other_tags))
print("Total tags:", len(tax_tags))

# Keep the original tax_chunks_full = [] and NUM reading code below.
tax_chunks_full = []

with ZipFile(notes_path) as zf:
    for chunk in pd.read_csv(
        zf.open("num.tsv"),
        sep="\t",
        dtype="string",
        chunksize=100_000
    ):
        matched = chunk.loc[
            chunk["tag"].isin(tax_tags)
        ].copy()

        if not matched.empty:
            tax_chunks_full.append(matched)

tax_all = pd.concat(tax_chunks_full, ignore_index=True)


Core tags: 4
Other tags: 4
Total tags: 8


In [3]:
# NUM-to-DIM lookup is a description of a scope, not an economic classification.
tax_with_dim = tax_all.merge(dim, left_on='dimh', right_on='dimhash', how='left', validate='many_to_one')
print(tax_all.groupby('tag', dropna=False).size())
assert tax_all['value'].notna().eq(pd.to_numeric(tax_all['value'], errors='coerce').notna()).all()


tag
IncomeTaxPaidFederalAfterRefundReceived               1436
IncomeTaxPaidForeignAfterRefundReceived               2049
IncomeTaxPaidOther                                       2
IncomeTaxPaidOtherAfterRefundReceived                    5
IncomeTaxPaidOtherAfterRefundReceivedTotal               1
IncomeTaxPaidOtherStateAndLocalAfterRefundReceived       1
IncomeTaxPaidStateAndLocalAfterRefundReceived         3437
IncomeTaxesPaidNet                                    4929
dtype: int64


## 3. Duplicate validation before iprx selection
Recompute all seven-key groups using Decimal arithmetic and declared precision.
Equal-decimal facts must agree; otherwise all intervals must share an intersection.
The chosen `iprx=0` is unique in every group, including singletons, for this pilot.
Preserve group footnotes and narrower joint intervals. This is a validated pilot
rule, not a universal SEC guarantee. Missing groups remain missing.


In [4]:
qa_groups, duplicate_facts = audit_duplicates(tax_all, fact_key)
tax_selected = tax_all.loc[tax_all['iprx'].eq('0')].copy()
assert not tax_selected.duplicated(fact_key).any()
assert len(tax_selected) == tax_all.groupby(fact_key, dropna=False).ngroups
core_raw_count = tax_all['tag'].isin(core_tax_tags).sum()
assert core_raw_count == 11851
assert len(tax_all) == 11860 and len(tax_selected) == 10926
assert len(qa_groups) == 919
print(qa_groups['decision'].value_counts())
qa_groups.to_csv(OUTPUT/'duplicate_group_qa.csv', index=False)
duplicate_facts.to_csv(OUTPUT/'duplicate_fact_audit.csv', index=False)
tax_all.to_csv(OUTPUT/'cash_tax_raw_selected_concepts.csv', index=False)


decision
KEEP_0_NUMERIC_PASS                894
KEEP_0_MISSING_NO_NUMERIC_CLAIM     15
KEEP_0_PRESERVE_ANNOTATIONS          8
KEEP_0_PRESERVE_INTERVAL             2
Name: count, dtype: int64


In [5]:
notes_to_keep = qa_groups.loc[
    qa_groups["decision"] == "KEEP_0_PRESERVE_ANNOTATIONS",
    fact_key + ["group_footnotes_json"]
].copy()

print(len(notes_to_keep))


8


In [6]:
intervals_to_keep = qa_groups.loc[
    qa_groups["decision"] == "KEEP_0_PRESERVE_INTERVAL",
    fact_key + ["intersection_low", "intersection_high"]
].copy()

for col in fact_key:
    intervals_to_keep[col] = (
        intervals_to_keep[col].astype("string")
    )

tax_selected = tax_selected.merge(
    intervals_to_keep.rename(
        columns={
            "intersection_low": "duplicate_interval_low",
            "intersection_high": "duplicate_interval_high"
        }
    ),
    on=fact_key,
    how="left",
    validate="one_to_one"
)

print(
    "Preserved interval groups:",
    tax_selected["duplicate_interval_low"].notna().sum()
)


Preserved interval groups: 2


In [7]:
for col in fact_key:
    notes_to_keep[col] = notes_to_keep[col].astype("string")

tax_selected = tax_selected.merge(
    notes_to_keep.rename(
        columns={"group_footnotes_json": "duplicate_footnotes_json"}
    ),
    on=fact_key,
    how="left",
    validate="one_to_one"
)


In [8]:
tax_selected['value_num'] = pd.to_numeric(tax_selected['value'], errors='coerce')
assert tax_selected['value_num'].isna().sum() == 463
print('Selected facts:', len(tax_selected), 'Missing:', tax_selected['value_num'].isna().sum())
tax_selected.to_csv(OUTPUT/'cash_tax_selected_all_periods.csv', index=False)


Selected facts: 10926 Missing: 463


## 4. Period and unit classification
Preserve the development pilot's mechanical definition: original `10-K`, `fp=FY`,
`qtrs=4`, and standardized NUM `ddate=SUB.period` for current candidates.
Prior comparisons and unusual standardized gaps are retained separately. `qtrs`
and `ddate` are standardized SEC fields, not exact original context dates.
Fiscal-year labels need not equal calendar years. No exclusion uses that mismatch alone.


In [9]:
period_check = tax_selected.merge(
    sub[["adsh", "form", "period", "fy", "fp"]],
    on="adsh",
    how="left",
    validate="many_to_one"
)


In [10]:
annual_candidates = period_check.loc[
    (period_check["form"] == "10-K")
    & (period_check["fp"] == "FY")
    & (period_check["qtrs"] == "4")
].copy()

print(len(annual_candidates))


9732


In [11]:
annual_candidates["ddate_month"] = (
    annual_candidates["ddate"].str[:6]
)

annual_candidates["period_month"] = (
    annual_candidates["period"].str[:6]
)

annual_candidates["period_relation"] = "REVIEW"

annual_candidates.loc[
    annual_candidates["ddate_month"] ==
    annual_candidates["period_month"],
    "period_relation"
] = "SAME_MONTH"

annual_candidates.loc[
    annual_candidates["ddate_month"] <
    annual_candidates["period_month"],
    "period_relation"
] = "EARLIER_MONTH"

annual_candidates.loc[
    annual_candidates["ddate_month"] >
    annual_candidates["period_month"],
    "period_relation"
] = "LATER_MONTH"

print(
    annual_candidates["period_relation"]
    .value_counts(dropna=False)
)


period_relation
SAME_MONTH       5548
EARLIER_MONTH    4184
Name: count, dtype: int64


In [12]:
annual_candidates["period_date"] = pd.to_datetime(
    annual_candidates["period"],
    format="%Y%m%d"
)

annual_candidates["ddate_date"] = pd.to_datetime(
    annual_candidates["ddate"],
    format="%Y%m%d"
)

annual_candidates["gap_days"] = (
    annual_candidates["period_date"]
    - annual_candidates["ddate_date"]
).dt.days

print(
    annual_candidates["gap_days"]
    .value_counts()
    .head(15)
)


gap_days
0      5548
365    2840
731    1339
549       1
486       1
610       1
457       1
92        1
Name: count, dtype: int64


In [13]:
annual_candidates["period_type"] = "COMPARATIVE_CANDIDATE"

annual_candidates.loc[
    annual_candidates["gap_days"] == 0,
    "period_type"
] = "CURRENT_CANDIDATE"

annual_candidates.loc[
    ~annual_candidates["gap_days"].isin([0, 365, 731]),
    "period_type"
] = "SPECIAL_COMPARATIVE_REVIEW"

print(annual_candidates["period_type"].value_counts())


period_type
CURRENT_CANDIDATE             5548
COMPARATIVE_CANDIDATE         4179
SPECIAL_COMPARATIVE_REVIEW       5
Name: count, dtype: int64


In [14]:
annual_candidates = annual_candidates.merge(
    sub[["adsh", "cik"]],
    on="adsh",
    how="left",
    validate="many_to_one"
)

print("Missing CIK:", annual_candidates["cik"].isna().sum())


Missing CIK: 0


In [15]:
annual_candidates["ddate_year"] = pd.to_numeric(
    annual_candidates["ddate"].str[:4],
    errors="coerce"
)

annual_candidates["fy_num"] = pd.to_numeric(
    annual_candidates["fy"],
    errors="coerce"
)

annual_candidates["fy_offset"] = (
    annual_candidates["fy_num"]
    - annual_candidates["ddate_year"]
)

display(
    pd.crosstab(
        annual_candidates["period_type"],
        annual_candidates["fy_offset"],
        margins=True
    )
)


fy_offset,-1,0,1,2,3,All
period_type,,,,,,
COMPARATIVE_CANDIDATE,0,126,2831,1220,2,4179
CURRENT_CANDIDATE,244,5264,40,0,0,5548
SPECIAL_COMPARATIVE_REVIEW,0,1,3,1,0,5
All,244,5391,2874,1221,2,9732


In [16]:
if "fye" not in annual_candidates.columns:
    annual_candidates = annual_candidates.merge(
        sub[["adsh", "fye"]].drop_duplicates("adsh"),
        on="adsh",
        how="left",
        validate="many_to_one"
    )

fy_label_cases = annual_candidates.loc[
    (annual_candidates["period_type"] == "CURRENT_CANDIDATE")
    & (annual_candidates["fy_offset"] == 1)
].drop_duplicates("adsh")

display(
    fy_label_cases[
        ["cik", "adsh", "fy", "fp", "fye", "period", "ddate"]
    ]
)


,cik,adsh,fy,fp,fye,period,ddate
1393,1278027,0001104659-26-022961,2026,FY,1231,20251231,20251231
2799,1024795,0001193125-26-087747,2026,FY,1231,20251231,20251231
2855,1856236,0001193125-26-091643,2026,FY,1231,20251231,20251231
3705,1832332,0001193125-26-115851,2026,FY,1231,20251231,20251231
8132,31107,0001654954-26-001850,2026,FY,1231,20251231,20251231


In [17]:
# 1. Define the four cash tax categories
# 1. Define the four core cash-tax categories

core_cash_tax_tag_map = {
    "IncomeTaxesPaidNet": "TOTAL",
    "IncomeTaxPaidFederalAfterRefundReceived": "FEDERAL",
    "IncomeTaxPaidStateAndLocalAfterRefundReceived": "STATE_LOCAL",
    "IncomeTaxPaidForeignAfterRefundReceived": "FOREIGN"
}

# 2. Preserve each custom Other tag as a separate category

other_cash_tax_tag_map = {
    tag: f"OTHER_CUSTOM::{tag}"
    for tag in other_tags
}

# 3. Merge core and custom tag mappings

cash_tax_tag_map = {
    **core_cash_tax_tag_map,
    **other_cash_tax_tag_map
}

print("Core categories:", len(core_cash_tax_tag_map))
print("Other categories:", len(other_cash_tax_tag_map))
print("Total mapped categories:", len(cash_tax_tag_map))

assert set(other_tags).issubset(cash_tax_tag_map.keys())

# 4. Extract all mapped tags

cash_tax_candidates = annual_candidates.loc[
    annual_candidates["tag"].isin(cash_tax_tag_map.keys())
].copy()

# 5. Create tax_type

cash_tax_candidates["tax_type"] = (
    cash_tax_candidates["tag"].map(cash_tax_tag_map)
)

# 6. QA

assert cash_tax_candidates["tax_type"].notna().all()

assert cash_tax_candidates["tag"].isin(
    cash_tax_tag_map.keys()
).all()

# 5. Inspect the results

print("Total candidate observations:", len(cash_tax_candidates))

print("\nObservations by tax type:")
print(cash_tax_candidates["tax_type"].value_counts())

print("\nTax type × period type:")
display(
    pd.crosstab(
        cash_tax_candidates["tax_type"],
        cash_tax_candidates["period_type"],
        margins=True
    )
)

print("\nTax type × currency:")
display(
    pd.crosstab(
        cash_tax_candidates["tax_type"],
        cash_tax_candidates["uom"],
        margins=True
    )
)


Core categories: 4
Other categories: 4
Total mapped categories: 8
Total candidate observations: 9732

Observations by tax type:
tax_type
TOTAL                                                               3345
STATE_LOCAL                                                         3254
FOREIGN                                                             1795
FEDERAL                                                             1329
OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceived                    5
OTHER_CUSTOM::IncomeTaxPaidOther                                       2
OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceivedTotal               1
OTHER_CUSTOM::IncomeTaxPaidOtherStateAndLocalAfterRefundReceived       1
Name: count, dtype: int64

Tax type × period type:


period_type,COMPARATIVE_CANDIDATE,CURRENT_CANDIDATE,SPECIAL_COMPARATIVE_REVIEW,All
tax_type,,,,
FEDERAL,538,791,0,1329
FOREIGN,531,1264,0,1795
OTHER_CUSTOM::IncomeTaxPaidOther,1,1,0,2
OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceived,2,3,0,5
OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceivedTotal,0,1,0,1
OTHER_CUSTOM::IncomeTaxPaidOtherStateAndLocalAfterRefundReceived,0,1,0,1
STATE_LOCAL,1331,1923,0,3254
TOTAL,1776,1564,5,3345
All,4179,5548,5,9732



Tax type × currency:


uom,CAD,USD,All
tax_type,,,
FEDERAL,0,1329,1329
FOREIGN,0,1795,1795
OTHER_CUSTOM::IncomeTaxPaidOther,0,2,2
OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceived,0,5,5
OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceivedTotal,0,1,1
OTHER_CUSTOM::IncomeTaxPaidOtherStateAndLocalAfterRefundReceived,0,1,1
STATE_LOCAL,0,3254,3254
TOTAL,2,3343,3345
All,2,9730,9732


In [18]:
# 1. Extract current-period candidates

current_candidates = cash_tax_candidates.loc[
    cash_tax_candidates["period_type"] == "CURRENT_CANDIDATE"
].copy()


# 2. Preserve comparative candidates

comparative_candidates = cash_tax_candidates.loc[
    cash_tax_candidates["period_type"] == "COMPARATIVE_CANDIDATE"
].copy()


# 3. Preserve special cases for review

special_candidates = cash_tax_candidates.loc[
    cash_tax_candidates["period_type"] == "SPECIAL_COMPARATIVE_REVIEW"
].copy()


# 4. Check the results

print("Current candidates:", len(current_candidates))

print("Comparative candidates:", len(comparative_candidates))

print("Special candidates:", len(special_candidates))

print("Original candidates:", len(cash_tax_candidates))


# 5. Verify that no observations were lost

assert (
    len(current_candidates)
    + len(comparative_candidates)
    + len(special_candidates)
    == len(cash_tax_candidates)
)


# 6. Inspect current candidates by tax type

print("\nCurrent candidates by tax type:")

print(
    current_candidates["tax_type"]
    .value_counts()
)


# 7. Inspect current candidates by currency

display(
    pd.crosstab(
        current_candidates["tax_type"],
        current_candidates["uom"],
        margins=True
    )
)


Current candidates:

 5548
Comparative candidates: 4179
Special candidates: 5
Original candidates: 9732

Current candidates by tax type:
tax_type
STATE_LOCAL                                                         1923
TOTAL                                                               1564
FOREIGN                                                             1264
FEDERAL                                                              791
OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceived                    3
OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceivedTotal               1
OTHER_CUSTOM::IncomeTaxPaidOtherStateAndLocalAfterRefundReceived       1
OTHER_CUSTOM::IncomeTaxPaidOther                                       1
Name: count, dtype: int64


uom,CAD,USD,All
tax_type,,,
FEDERAL,0,791,791
FOREIGN,0,1264,1264
OTHER_CUSTOM::IncomeTaxPaidOther,0,1,1
OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceived,0,3,3
OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceivedTotal,0,1,1
OTHER_CUSTOM::IncomeTaxPaidOtherStateAndLocalAfterRefundReceived,0,1,1
STATE_LOCAL,0,1923,1923
TOTAL,1,1563,1564
All,1,5547,5548


In [19]:
# 1. Create the unified current-period fact table

cash_tax_current = current_candidates.copy()


# 2. Define required columns

required_columns = [
    "adsh",
    "tag",
    "tax_type",
    "period_type",
    "uom",
    "value"
]


# 3. Check whether required columns exist

missing_columns = [
    col
    for col in required_columns
    if col not in cash_tax_current.columns
]

print("Missing required columns:", missing_columns)

assert len(missing_columns) == 0


# 4. Verify current-period classification

assert (
    cash_tax_current["period_type"]
    .eq("CURRENT_CANDIDATE")
    .all()
)


# 5. Verify tax type classification

valid_tax_types = set(cash_tax_tag_map.values())

assert (
    cash_tax_current["tax_type"]
    .notna()
    .all()
)

assert (
    set(cash_tax_current["tax_type"].unique())
    .issubset(valid_tax_types)
)


# 6. Check the final dataset

print("Final current-period observations:", len(cash_tax_current))

print("\nTax type distribution:")

print(
    cash_tax_current["tax_type"]
    .value_counts()
)

print("\nCurrency distribution:")

print(
    cash_tax_current["uom"]
    .value_counts(dropna=False)
)


Missing required columns: []
Final current-period observations: 5548

Tax type distribution:
tax_type
STATE_LOCAL                                                         1923
TOTAL                                                               1564
FOREIGN                                                             1264
FEDERAL                                                              791
OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceived                    3
OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceivedTotal               1
OTHER_CUSTOM::IncomeTaxPaidOtherStateAndLocalAfterRefundReceived       1
OTHER_CUSTOM::IncomeTaxPaidOther                                       1
Name: count, dtype: int64

Currency distribution:
uom
USD    5547
CAD       1
Name: count, dtype: Int64


In [20]:
# Convert raw values into numeric values

cash_tax_current["value_num"] = pd.to_numeric(
    cash_tax_current["value"],
    errors="coerce"
)


# Check conversion results

print("Original missing:",
      cash_tax_current["value"].isna().sum())

print("Numeric missing:",
      cash_tax_current["value_num"].isna().sum())

print("Conversion failures:",
      (
          cash_tax_current["value"].notna()
          & cash_tax_current["value_num"].isna()
      ).sum())

print("Numeric dtype:",
      cash_tax_current["value_num"].dtype)


Original missing: 179
Numeric missing: 179
Conversion failures: 0
Numeric dtype: Float64


In [21]:
comparative_candidates.to_csv(OUTPUT/'cash_tax_comparative_facts.csv', index=False)
special_candidates.to_csv(OUTPUT/'cash_tax_special_period_facts.csv', index=False)


## 5. Dimensions and jurisdiction diagnostics
DIM joins are many-to-one. Tax categories come from concepts; dimensional axes
describe reporting scope. California/Canada candidates use tag context, and
DomesticCountry inference requires agreeing SUB business/incorporation countries.
These are diagnostic labels, not verified geographic allocation amounts.


In [22]:
# Extract the existing dimension information

dim_lookup = tax_with_dim[
    ["dimhash", "segments", "segt"]
].drop_duplicates()


# Check whether each dimhash has only one mapping

print("Dimension mapping rows:", len(dim_lookup))

print(
    "Conflicting dimhash:",
    dim_lookup["dimhash"].duplicated().sum()
)

assert dim_lookup["dimhash"].duplicated().sum() == 0


Dimension mapping rows: 575
Conflicting dimhash: 0


In [23]:
cash_tax_current = cash_tax_current.merge(
    dim_lookup,
    left_on="dimh",
    right_on="dimhash",
    how="left",
    validate="many_to_one",
    indicator="dim_match"
)


In [24]:
# Extract IncomeTaxAuthority and Geographical from segments

cash_tax_current["income_tax_authority_member"] = (
    cash_tax_current["segments"]
    .str.extract(
        r"(?:^|;)IncomeTaxAuthority=([^;]+)",
        expand=False
    )
)

cash_tax_current["geographical_member"] = (
    cash_tax_current["segments"]
    .str.extract(
        r"(?:^|;)Geographical=([^;]+)",
        expand=False
    )
)


In [25]:
# Whether any dimension exists
cash_tax_current["has_segments"] = (
    cash_tax_current["segments"].notna()
)

# Whether IncomeTaxAuthority exists
cash_tax_current["has_income_tax_authority"] = (
    cash_tax_current["income_tax_authority_member"].notna()
)

# Whether Geographical exists
cash_tax_current["has_geographical"] = (
    cash_tax_current["geographical_member"].notna()
)


# Compare the three indicators
for col in [
    "has_segments",
    "has_income_tax_authority",
    "has_geographical"
]:
    print("\n", col)
    print(
        cash_tax_current[col].value_counts()
    )



 has_segments
has_segments
False    2814
True     2734
Name: count, dtype: int64

 has_income_tax_authority
has_income_tax_authority
False    3282
True     2266
Name: count, dtype: int64

 has_geographical
has_geographical
False    5373
True      175
Name: count, dtype: int64


In [26]:
# Use actual authority members observed in the data

cash_tax_current["authority_class"] = "REVIEW"

m = cash_tax_current["income_tax_authority_member"]
t = cash_tax_current["tax_type"]

# No authority dimension
cash_tax_current.loc[
    m.isna(),
    "authority_class"
] = "NO_AUTHORITY"

# Broad categories: useful even without a specific jurisdiction
cash_tax_current.loc[
    m.isin([
        "StateAndLocalTaxJurisdictionOther",
        "StateAndLocalJurisdiction"
    ]),
    "authority_class"
] = "BROAD_STATE_LOCAL"

cash_tax_current.loc[
    m.isin([
        "ForeignTaxJurisdictionOther",
        "ForeignCountry"
    ]),
    "authority_class"
] = "BROAD_FOREIGN"

# Context-based candidates: CA is ambiguous by itself
cash_tax_current.loc[
    m.eq("CA") & t.eq("STATE_LOCAL"),
    "authority_class"
] = "CALIFORNIA_CANDIDATE"

cash_tax_current.loc[
    m.eq("CA") & t.eq("FOREIGN"),
    "authority_class"
] = "CANADA_CANDIDATE"

# Inspect
display(
    pd.crosstab(
        cash_tax_current["tax_type"],
        cash_tax_current["authority_class"],
        margins=True
    )
)


authority_class,BROAD_FOREIGN,BROAD_STATE_LOCAL,CALIFORNIA_CANDIDATE,CANADA_CANDIDATE,NO_AUTHORITY,REVIEW,All
tax_type,,,,,,,
FEDERAL,0,0,0,0,762,29,791
FOREIGN,178,2,0,80,399,605,1264
OTHER_CUSTOM::IncomeTaxPaidOther,0,0,0,0,1,0,1
OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceived,0,0,0,0,2,1,3
OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceivedTotal,0,0,0,0,1,0,1
OTHER_CUSTOM::IncomeTaxPaidOtherStateAndLocalAfterRefundReceived,0,0,0,0,1,0,1
STATE_LOCAL,6,279,105,0,746,787,1923
TOTAL,19,38,0,0,1370,137,1564
All,203,319,105,80,3282,1559,5548


In [27]:
# 1.4 — Specific jurisdiction candidates (pilot v1)

# Capture an additional authority-name axis
cash_tax_current["authority_name_member"] = (
    cash_tax_current["segments"].str.extract(
        r"(?:^|;)IncomeTaxAuthorityName=([^;]+)",
        expand=False
    )
)

# Start with unresolved jurisdictions
cash_tax_current["jurisdiction_id"] = pd.Series(
    pd.NA, index=cash_tax_current.index, dtype="string"
)
cash_tax_current["jurisdiction_source"] = "UNRESOLVED"

# Reference sets (initial version)
us_states = set(
    "AL AK AZ AR CA CO CT DE FL GA HI ID IL IN IA KS KY LA "
    "ME MD MA MI MN MS MO MT NE NV NH NJ NM NY NC ND OH "
    "OK OR PA RI SC SD TN TX UT VT VA WA WV WI WY DC".split()
)

foreign_countries = set(
    "CA GB IN IL NL MX CN BR AU HK ES DE CO LA MA PA "
    "SG JP KR FR IE CH IT SE NO DK BE ZA NZ TW".split()
)

# First use IncomeTaxAuthority; then Geographical as fallback
for col in ["income_tax_authority_member", "geographical_member"]:

    member = cash_tax_current[col]
    tax_type = cash_tax_current["tax_type"]
    unresolved = cash_tax_current["jurisdiction_id"].isna()

    state_mask = (
        unresolved
        & tax_type.eq("STATE_LOCAL")
        & member.isin(us_states)
    )

    foreign_mask = (
        unresolved
        & tax_type.eq("FOREIGN")
        & member.isin(foreign_countries)
    )

    cash_tax_current.loc[state_mask, "jurisdiction_id"] = (
        "US-" + member.loc[state_mask]
    )

    cash_tax_current.loc[foreign_mask, "jurisdiction_id"] = (
        member.loc[foreign_mask]
    )

    cash_tax_current.loc[
        state_mask | foreign_mask, "jurisdiction_source"
    ] = col + "_TAG_CONTEXT"

# Explicitly named tax authority, if not already classified
named_ca = (
    cash_tax_current["authority_name_member"]
    .eq("CaliforniaFranchiseTaxBoard")
    .fillna(False)
)

named_ca = (
    named_ca
    & cash_tax_current["jurisdiction_id"].isna()
    & cash_tax_current["tax_type"].isin(["STATE_LOCAL", "TOTAL"])
)

cash_tax_current.loc[named_ca, "jurisdiction_id"] = "US-CA"
cash_tax_current.loc[named_ca, "jurisdiction_source"] = (
    "NAMED_AUTHORITY_CANDIDATE"
)

# Inspect results
print("Jurisdiction identification sources:")
print(cash_tax_current["jurisdiction_source"].value_counts())

print("\nTop jurisdiction candidates:")
print(cash_tax_current["jurisdiction_id"].value_counts().head(20))


Jurisdiction identification sources:
jurisdiction_source
UNRESOLVED                                 4043
income_tax_authority_member_TAG_CONTEXT    1387
geographical_member_TAG_CONTEXT              95
NAMED_AUTHORITY_CANDIDATE                    23
Name: count, dtype: int64

Top jurisdiction candidates:
jurisdiction_id
US-CA    135
US-TX    120
CA        88
US-NY     87
GB        77
US-PA     58
IN        54
US-FL     52
US-NJ     48
US-IL     47
US-MA     44
DE        44
NL        40
MX        37
CN        36
BR        30
US-TN     28
AU        25
US-NC     21
FR        20
Name: count, dtype: Int64


In [28]:
# 1. Extract IncomeTaxAuthorityName separately

cash_tax_current["authority_name_member"] = (
    cash_tax_current["segments"]
    .astype("string")
    .str.extract(
        r"(?:^|;)IncomeTaxAuthorityName=([^;]+)",
        expand=False
    )
)

print("IncomeTaxAuthorityName distribution:")

print(
    cash_tax_current["authority_name_member"]
    .value_counts(dropna=False)
)

# 2. Prepare SUB address information

address_cols = [
    "countryba",
    "countryinc",
    "stprba",
    "stprinc"
]

required_cols = ["adsh"] + address_cols

missing_cols = [
    col for col in required_cols
    if col not in sub.columns
]

print("\nMissing SUB columns:", missing_cols)

assert not missing_cols, "Required SUB columns are missing"

sub_address = sub[required_cols].copy()

# 3. Check whether adsh is unique in SUB

duplicate_sub = sub_address[
    sub_address["adsh"].duplicated(keep=False)
]

print("Duplicate SUB accession rows:", len(duplicate_sub))

assert duplicate_sub.empty, "SUB adsh is not unique"

# 4. Merge SUB information into current tax facts

rows_before = len(cash_tax_current)

cash_tax_current["adsh"] = (
    cash_tax_current["adsh"].astype("string")
)

sub_address["adsh"] = (
    sub_address["adsh"].astype("string")
)

# Refresh old address columns if they already exist
cash_tax_current = cash_tax_current.drop(
    columns=[
        col for col in address_cols
        if col in cash_tax_current.columns
    ]
)

cash_tax_current = cash_tax_current.merge(
    sub_address,
    on="adsh",
    how="left",
    validate="many_to_one",
    indicator="_sub_match"
)

print("\nSUB matching:")
print(cash_tax_current["_sub_match"].value_counts())

print("\nMissing address information:")
print(cash_tax_current[address_cols].isna().sum())

print("\nRows before:", rows_before)
print("Rows after:", len(cash_tax_current))

assert len(cash_tax_current) == rows_before


IncomeTaxAuthorityName distribution:
authority_name_member
<NA>                                        5285
CaliforniaFranchiseTaxBoard                   23
NewYorkStateDivisionOfTaxationAndFinance      14
InternalRevenueServiceIRS                     11
NewJerseyDivisionOfTaxation                    9
                                            ... 
Canada                                         1
Wisconsin                                      1
China                                          1
OtherCountries                                 1
Mexico                                         1
Name: count, Length: 152, dtype: Int64

Missing SUB columns: []
Duplicate SUB accession rows: 0

SUB matching:
_sub_match
both          5548
left_only        0
right_only       0
Name: count, dtype: int64

Missing address information:
countryba      71
countryinc    424
stprba        149
stprinc       566
dtype: int64

Rows before: 5548
Rows after: 5548


In [29]:
# 1. Find DomesticCountry facts
domestic_mask = (
    cash_tax_current["income_tax_authority_member"]
    .eq("DomesticCountry")
    .fillna(False)
)

# 2. Get the two country fields
ba = cash_tax_current["countryba"].astype("string").str.strip().str.upper()
inc = cash_tax_current["countryinc"].astype("string").str.strip().str.upper()

# 3. Check whether both country codes agree
same_country = (
    ba.notna()
    & inc.notna()
    & ba.eq(inc).fillna(False)
)

# 4. Create an empty inference column
cash_tax_current["inferred_country_id"] = pd.Series(
    pd.NA,
    index=cash_tax_current.index,
    dtype="string"
)

# 5. Fill only eligible DomesticCountry facts
inference_mask = domestic_mask & same_country

cash_tax_current.loc[
    inference_mask, "inferred_country_id"
] = ba.loc[inference_mask]

# 6. Inspect
print("DomesticCountry facts:", domestic_mask.sum())
print("Address-agreement candidates:", inference_mask.sum())

display(
    cash_tax_current.loc[
        domestic_mask,
        ["tax_type", "countryba", "countryinc", "inferred_country_id"]
    ].head(15)
)


DomesticCountry facts: 36
Address-agreement candidates: 32


,tax_type,countryba,countryinc,inferred_country_id
136,TOTAL,US,US,US
173,TOTAL,US,US,US
242,TOTAL,US,US,US
573,TOTAL,US,US,US
788,TOTAL,US,US,US
868,FEDERAL,US,US,US
1383,FEDERAL,US,US,US
1522,FEDERAL,US,US,US
1536,FEDERAL,PR,<NA>,<NA>
1657,TOTAL,US,US,US


In [30]:
cash_tax_current.to_csv(OUTPUT/'cash_tax_facts_current_long.csv', index=False)
assert len(cash_tax_current) == 5548
assert cash_tax_current['segt'].eq('0').all(), 'Truncated DIM text requires review before axis classification.'


## 6. A — Strict reconciliation
Match the same accession, standardized end date, duration, unit and complete
dimension hash. A complete scope has four numeric core categories; absence/nil is
not zero. Test Total minus the three components without adding geographic details.
The legacy 184 scopes and all seven original nonzero differences must reproduce.


In [31]:
# Convert dimension count to numeric

dimn_num = pd.to_numeric(
    cash_tax_current["dimn"],
    errors="coerce"
)

# Select dimensionless USD facts

base_facts = cash_tax_current.loc[
    dimn_num.eq(0)
    & cash_tax_current["uom"].eq("USD")
    & cash_tax_current["value_num"].notna()
].copy()

print("Eligible base facts:", len(base_facts))

assert len(base_facts) > 0

key = ["adsh", "ddate", "qtrs", "tax_type"]

fact_counts = (
    base_facts.groupby(key, dropna=False)
    .size()
)

print("Duplicate groups:", (fact_counts > 1).sum())

display(fact_counts[fact_counts > 1])


Eligible base facts: 2671
Duplicate groups: 0


Series([], dtype: int64)

In [32]:
# 1.5 — Identify comparable reporting scopes

scope_key = [
    "adsh",
    "ddate",
    "qtrs",
    "uom",
    "dimh"
]

scope_counts = (
    cash_tax_current
    .groupby(scope_key, dropna=False)["tax_type"]
    .nunique()
)

print("Reporting scopes:", len(scope_counts))

print(
    "Scopes with 2+ tax types:",
    (scope_counts >= 2).sum()
)

print(
    "Scopes with 3+ tax types:",
    (scope_counts >= 3).sum()
)

core_tax_types = [
    "TOTAL",
    "FEDERAL",
    "STATE_LOCAL",
    "FOREIGN"
]

core_scope_counts = (
    cash_tax_current.loc[
        cash_tax_current["tax_type"].isin(core_tax_types)
    ]
    .groupby(scope_key, dropna=False)["tax_type"]
    .nunique()
)

print(
    "Scopes with all 4 core tax types:",
    (core_scope_counts == 4).sum()
)


Reporting scopes: 4012
Scopes with 2+ tax types: 809
Scopes with 3+ tax types: 528
Scopes with all 4 core tax types: 196


In [33]:
# 1.5A — Strict reconciliation
pivot_key = scope_key + ["tax_type"]

duplicate_pivot_facts = cash_tax_current.loc[
    cash_tax_current.duplicated(
        subset=pivot_key,
        keep=False
    )
].sort_values(pivot_key)

print(
    "Duplicate scope-tax_type facts:",
    len(duplicate_pivot_facts)
)

display(duplicate_pivot_facts)

assert duplicate_pivot_facts.empty, (
    "Duplicate facts detected. Review before pivoting; "
    "do not automatically sum them."
)
wide = cash_tax_current.pivot(
    index=scope_key,
    columns="tax_type",
    values="value_num"
)

# Keep scopes with all four numeric amounts
four_types = ["TOTAL", "FEDERAL", "STATE_LOCAL", "FOREIGN"]

complete = wide.dropna(subset=four_types).copy()

# Calculate the sum of components
complete["sum_parts"] = (
    complete["FEDERAL"]
    + complete["STATE_LOCAL"]
    + complete["FOREIGN"]
)

# Calculate reconciliation difference
complete["difference"] = (
    complete["TOTAL"] - complete["sum_parts"]
)

print("Complete numeric scopes:", len(complete))

print("\nExact matches:")
print(complete["difference"].eq(0).sum())

print("\nDifference summary:")
display(complete["difference"].describe())

display(complete.head(10))


Duplicate scope-tax_type facts: 0


,adsh,tag,version,ddate,qtrs,uom,dimh,iprx,value,footnote,...,authority_class,authority_name_member,jurisdiction_id,jurisdiction_source,countryba,countryinc,stprba,stprinc,_sub_match,inferred_country_id


Complete numeric scopes:

 184

Exact matches:
177

Difference summary:


count            184.0
mean      28347.826087
std      289726.154284
min         -1000000.0
25%                0.0
50%                0.0
75%                0.0
max          3444000.0
Name: difference, dtype: Float64

,,,,tax_type,FEDERAL,FOREIGN,OTHER_CUSTOM::IncomeTaxPaidOther,OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceived,OTHER_CUSTOM::IncomeTaxPaidOtherAfterRefundReceivedTotal,OTHER_CUSTOM::IncomeTaxPaidOtherStateAndLocalAfterRefundReceived,STATE_LOCAL,TOTAL,sum_parts,difference
adsh,ddate,qtrs,uom,dimh,,,,,,,,,,
0000080035-26-000007,20251231,4,USD,0x00000000,1909000.0,9869000.0,<NA>,<NA>,<NA>,<NA>,-353000.0,11425000.0,11425000.0,0.0
0000091388-26-000014,20251231,4,USD,0x00000000,103000000.0,3000000.0,<NA>,<NA>,<NA>,<NA>,19000000.0,124000000.0,125000000.0,-1000000.0
0000104169-26-000055,20260131,4,USD,0x00000000,1743000000.0,2726000000.0,<NA>,<NA>,<NA>,<NA>,895000000.0,5364000000.0,5364000000.0,0.0
0000320121-26-000010,20251231,4,USD,0x00000000,0.0,12000.0,<NA>,<NA>,<NA>,<NA>,30000.0,42000.0,42000.0,0.0
0000352825-26-000016,20251231,4,USD,0x00000000,290000.0,996000.0,<NA>,<NA>,<NA>,<NA>,345000.0,1631000.0,1631000.0,0.0
0000701985-26-000008,20260131,4,USD,0x00000000,177000000.0,10000000.0,<NA>,<NA>,<NA>,<NA>,36000000.0,223000000.0,223000000.0,0.0
0000703604-26-000008,20251231,4,USD,0x00000000,4897000.0,10416000.0,<NA>,<NA>,<NA>,<NA>,2874000.0,18187000.0,18187000.0,0.0
0000708781-26-000010,20251231,4,USD,0x00000000,4103000.0,98000.0,<NA>,<NA>,<NA>,<NA>,935000.0,5136000.0,5136000.0,0.0
0000711669-26-000007,20251231,4,USD,0x00000000,4210000.0,0.0,<NA>,<NA>,<NA>,<NA>,226000.0,4436000.0,4436000.0,0.0


In [34]:
# 1.5A — Inspect the seven reconciliation exceptions

# 1. Extract the seven mismatching scopes
mismatch_scopes = (
    complete.loc[
        complete["difference"].ne(0)
    ]
    .reset_index()
    .copy()
)

print("Mismatching scopes:", len(mismatch_scopes))

# 2. Retrieve their original tax facts
mismatch_detail = cash_tax_current.merge(
    mismatch_scopes,
    on=scope_key,
    how="inner",
    validate="many_to_one"
)

# 3. Display the original facts and precision information
display(
    mismatch_detail[
        [
            "adsh",
            "tax_type",
            "tag",
            "value",
            "value_num",
            "dcml",
            "iprx",
            "dimn",
            "segments",
            "footnote",
            "difference"
        ]
    ].sort_values(
        ["adsh", "tax_type"]
    )
)


Mismatching scopes: 7


,adsh,tax_type,tag,value,value_num,dcml,iprx,dimn,segments,footnote,difference
1,0000091388-26-000014,FEDERAL,IncomeTaxPaidFederalAfterRefundReceived,103000000.0000,103000000.0,-6,0,0,<NA>,<NA>,-1000000.0
2,0000091388-26-000014,FOREIGN,IncomeTaxPaidForeignAfterRefundReceived,3000000.0000,3000000.0,-6,0,0,<NA>,<NA>,-1000000.0
3,0000091388-26-000014,STATE_LOCAL,IncomeTaxPaidStateAndLocalAfterRefundReceived,19000000.0000,19000000.0,-6,0,0,<NA>,<NA>,-1000000.0
0,0000091388-26-000014,TOTAL,IncomeTaxesPaidNet,124000000.0000,124000000.0,-6,0,0,<NA>,<NA>,-1000000.0
5,0001104659-26-032065,FEDERAL,IncomeTaxPaidFederalAfterRefundReceived,131000.0000,131000.0,-3,0,0,<NA>,<NA>,726000.0
6,0001104659-26-032065,FOREIGN,IncomeTaxPaidForeignAfterRefundReceived,183000.0000,183000.0,-3,0,0,<NA>,<NA>,726000.0
7,0001104659-26-032065,STATE_LOCAL,IncomeTaxPaidStateAndLocalAfterRefundReceived,171000.0000,171000.0,-3,0,0,<NA>,<NA>,726000.0
4,0001104659-26-032065,TOTAL,IncomeTaxesPaidNet,1211000.0000,1211000.0,-3,0,0,<NA>,<NA>,726000.0
9,0001193125-26-104063,FEDERAL,IncomeTaxPaidFederalAfterRefundReceived,4050000.0000,4050000.0,-3,0,0,<NA>,<NA>,1287000.0
10,0001193125-26-104063,FOREIGN,IncomeTaxPaidForeignAfterRefundReceived,-209000.0000,-209000.0,-3,0,0,<NA>,<NA>,1287000.0


## 7. Extended reconciliation, CAL and B — Geographic diagnostics
Extended reconciliation considers separately disclosed CAL-linked Other components
and possible DIM relationships. It is distinct from precision compatibility.
The two filing-specific Other facts match tag/version and scope, but their CAL arcs
have negative weights. They are numerical candidates, not accepted additions. Geographic numerical gap
matches remain candidates. Partial disclosures are analyzed separately.

Smithfield is `ROUNDING_COMPATIBLE`, not confirmed rounding. California is part of
State and is not added again. BigBear.ai's original Total zero is preserved; its
legacy-row scope is semantically ambiguous and is excluded from analytical Total.


In [35]:
from zipfile import ZipFile

# The seven filings with reconciliation differences
wanted = set(mismatch_scopes["adsh"].astype("string"))

# Find CAL in the existing Notes ZIP
with ZipFile(notes_path) as zf:

    cal_name = next(
        (name for name in zf.namelist()
         if name.lower().endswith("cal.tsv")),
        None
    )

    print("CAL file:", cal_name)

    if cal_name is not None:

        parts = []

        for chunk in pd.read_csv(
            zf.open(cal_name),
            sep="\t",
            dtype={"adsh": "string"},
            chunksize=100000
        ):
            matched = chunk.loc[
                chunk["adsh"].isin(wanted)
            ]

            if not matched.empty:
                parts.append(matched)

        cal7 = (
            pd.concat(parts, ignore_index=True)
            if parts else pd.DataFrame()
        )

        print("CAL rows for seven filings:", len(cal7))

        if not cal7.empty:

            target_tags = set(cash_tax_current["tag"])

            tax_cal = cal7.loc[
                cal7["ptag"].isin(target_tags)
                | cal7["ctag"].isin(target_tags)
            ]

            print("Cash-tax calculation arcs:", len(tax_cal))

            display(
                tax_cal[
                    ["adsh", "grp", "arc",
                     "negative", "ptag", "ctag"]
                ].head(40)
            )


CAL file: cal.tsv


CAL rows for seven filings: 1450
Cash-tax calculation arcs: 20


,adsh,grp,arc,negative,ptag,ctag
76,0001493152-26-008465,12,1,1,IncomeTaxesPaidNet,IncomeTaxPaidFederalAfterRefundReceived
77,0001493152-26-008465,12,3,1,IncomeTaxesPaidNet,IncomeTaxPaidForeignAfterRefundReceived
78,0001493152-26-008465,12,2,1,IncomeTaxesPaidNet,IncomeTaxPaidStateAndLocalAfterRefundReceived
156,0001493152-26-008465,12,4,1,IncomeTaxesPaidNet,IncomeTaxPaidOtherAfterRefundReceived
543,0000091388-26-000014,22,1,1,IncomeTaxesPaidNet,IncomeTaxPaidFederalAfterRefundReceived
544,0000091388-26-000014,22,3,1,IncomeTaxesPaidNet,IncomeTaxPaidForeignAfterRefundReceived
545,0000091388-26-000014,22,2,1,IncomeTaxesPaidNet,IncomeTaxPaidStateAndLocalAfterRefundReceived
775,0001104659-26-032065,10,1,1,IncomeTaxesPaidNet,IncomeTaxPaidFederalAfterRefundReceived
776,0001104659-26-032065,10,3,1,IncomeTaxesPaidNet,IncomeTaxPaidForeignAfterRefundReceived
777,0001104659-26-032065,10,2,1,IncomeTaxesPaidNet,IncomeTaxPaidStateAndLocalAfterRefundReceived


In [36]:
# Full signed calculation evidence; retain all arcs for the seven filings.
cal7.to_csv(OUTPUT/'cal_arcs_seven_filings.csv', index=False)
display(tax_cal.loc[tax_cal['ctag'].str.contains('IncomeTaxPaidOther',na=False),
    ['adsh','grp','arc','negative','ptag','pversion','ctag','cversion']])


,adsh,grp,arc,negative,ptag,pversion,ctag,cversion
156,0001493152-26-008465,12,4,1,IncomeTaxesPaidNet,us-gaap/2025,IncomeTaxPaidOtherAfterRefundReceived,0001493152-26-008465
1255,0001193125-26-115851,8,36,1,IncomeTaxesPaidNet,us-gaap/2025,IncomeTaxPaidOtherStateAndLocalAfterRefundRece...,0001193125-26-115851


In [37]:
# Preserve the development Other diagnostic, correcting its interpretation.
cal_other_tag_map = {
    '0001193125-26-115851': 'IncomeTaxPaidOtherStateAndLocalAfterRefundReceived',
    '0001493152-26-008465': 'IncomeTaxPaidOtherAfterRefundReceived'}
expected_tag = cash_tax_current['adsh'].map(cal_other_tag_map)
cal_other_facts = cash_tax_current.loc[expected_tag.notna() & cash_tax_current['tag'].eq(expected_tag)].copy()
assert cal_other_facts.groupby('adsh').size().eq(1).all()
other_for_merge = cal_other_facts[scope_key+['tag','value_num']].rename(
    columns={'tag':'cal_other_tag','value_num':'other_candidate_amount'})
extended_difference_check = mismatch_scopes[scope_key+['difference']].merge(
    other_for_merge,on=scope_key,how='left',validate='one_to_one')
extended_difference_check['other_candidate_amount'] = extended_difference_check['other_candidate_amount'].fillna(0)
extended_difference_check['candidate_difference_after_other'] = (
    extended_difference_check['difference']-extended_difference_check['other_candidate_amount'])
display(extended_difference_check)
print('Numerical candidate only: CAL weights are checked before any accepted adjustment.')


,adsh,ddate,qtrs,uom,dimh,difference,cal_other_tag,other_candidate_amount,candidate_difference_after_other
0,0000091388-26-000014,20251231,4,USD,0x00000000,-1000000.0,<NA>,0.0,-1000000.0
1,0001104659-26-032065,20251231,4,USD,0x00000000,726000.0,<NA>,0.0,726000.0
2,0001193125-26-104063,20260131,4,USD,0x00000000,1287000.0,<NA>,0.0,1287000.0
3,0001193125-26-115851,20251231,4,USD,0x00000000,3444000.0,IncomeTaxPaidOtherStateAndLocalAfterRefundRece...,3444000.0,0.0
4,0001437749-26-010731,20251231,4,USD,0x00000000,679000.0,<NA>,0.0,679000.0
5,0001493152-26-008465,20251231,4,USD,0x00000000,52000.0,IncomeTaxPaidOtherAfterRefundReceived,52000.0,0.0
6,0001836981-26-000018,20251231,4,USD,0x00000000,28000.0,<NA>,0.0,28000.0


Numerical candidate only: CAL weights are checked before any accepted adjustment.


In [38]:
# A1.5.1A-6 — Step 1: Inspect geographic tax facts

geo_accessions = {
    "ONE Group": "0001104659-26-032065",
    "Zumiez": "0001193125-26-104063",
    "American Shared": "0001437749-26-010731"
}

# Match the original exception filing and reporting period
period_cols = ["adsh", "ddate", "qtrs", "uom"]

geo_scopes = mismatch_scopes.loc[
    mismatch_scopes["adsh"].astype("string").isin(
        geo_accessions.values()
    ),
    period_cols
].drop_duplicates().copy()

geo_facts = cash_tax_current.copy()

for col in period_cols:
    geo_scopes[col] = geo_scopes[col].astype("string")
    geo_facts[col] = geo_facts[col].astype("string")

# Deliberately exclude dimh from merge keys:
# we want to see ALL geographic dimensions in the same period.
geo_review = geo_facts.merge(
    geo_scopes,
    on=period_cols,
    how="inner",
    validate="many_to_one"
)

show_cols = [
    "adsh", "tax_type", "tag",
    "value", "dimh", "segments"
]

show_cols = [
    col for col in show_cols
    if col in geo_review.columns
]

for company, accession in geo_accessions.items():
    result = geo_review.loc[
        geo_review["adsh"].eq(accession),
        show_cols
    ].copy()

    print(f"\n{company}: {len(result)} tax facts")
    display(result.sort_values(["tax_type", "dimh"]))



ONE Group: 10 tax facts


,adsh,tax_type,tag,value,dimh,segments
1,0001104659-26-032065,FEDERAL,IncomeTaxPaidFederalAfterRefundReceived,131000.0000,0x00000000,<NA>
2,0001104659-26-032065,FOREIGN,IncomeTaxPaidForeignAfterRefundReceived,183000.0000,0x00000000,<NA>
3,0001104659-26-032065,STATE_LOCAL,IncomeTaxPaidStateAndLocalAfterRefundReceived,171000.0000,0x00000000,<NA>
4,0001104659-26-032065,STATE_LOCAL,IncomeTaxPaidStateAndLocalAfterRefundReceived,49000.0000,0x2826cc95da45f68d20fa84533a2159fb,IncomeTaxAuthority=MA;
5,0001104659-26-032065,STATE_LOCAL,IncomeTaxPaidStateAndLocalAfterRefundReceived,116000.0000,0x2fe48ac55bcad17c0b64e5dbe2f02f4c,IncomeTaxAuthority=NY;
6,0001104659-26-032065,STATE_LOCAL,IncomeTaxPaidStateAndLocalAfterRefundReceived,80000.0000,0x3275e5376030fcd92e79bd5aa0567803,IncomeTaxAuthority=FL;
7,0001104659-26-032065,STATE_LOCAL,IncomeTaxPaidStateAndLocalAfterRefundReceived,66000.0000,0x7266a8beee28957a9dd1231ff4de5594,IncomeTaxAuthority=PA;
8,0001104659-26-032065,STATE_LOCAL,IncomeTaxPaidStateAndLocalAfterRefundReceived,109000.0000,0xc7f56958c42c5923b575b69d4df18bf7,IncomeTaxAuthority=MN;
9,0001104659-26-032065,STATE_LOCAL,IncomeTaxPaidStateAndLocalAfterRefundReceived,306000.0000,0xfe1c91c2ce2a42631811c2e66def7b2e,IncomeTaxAuthority=TX;
0,0001104659-26-032065,TOTAL,IncomeTaxesPaidNet,1211000.0000,0x00000000,<NA>



Zumiez: 6 tax facts


,adsh,tax_type,tag,value,dimh,segments
11,0001193125-26-104063,FEDERAL,IncomeTaxPaidFederalAfterRefundReceived,4050000.0000,0x00000000,<NA>
12,0001193125-26-104063,FOREIGN,IncomeTaxPaidForeignAfterRefundReceived,-209000.0000,0x00000000,<NA>
13,0001193125-26-104063,FOREIGN,IncomeTaxPaidForeignAfterRefundReceived,792000.0000,0xa0f835c69d2115d529fe1aa84c96dd7b,IncomeTaxAuthority=CA;
14,0001193125-26-104063,STATE_LOCAL,IncomeTaxPaidStateAndLocalAfterRefundReceived,748000.0000,0x00000000,<NA>
15,0001193125-26-104063,STATE_LOCAL,IncomeTaxPaidStateAndLocalAfterRefundReceived,495000.0000,0xa0f835c69d2115d529fe1aa84c96dd7b,IncomeTaxAuthority=CA;
10,0001193125-26-104063,TOTAL,IncomeTaxesPaidNet,5876000.0000,0x00000000,<NA>



American Shared: 7 tax facts


,adsh,tax_type,tag,value,dimh,segments
17,0001437749-26-010731,FEDERAL,IncomeTaxPaidFederalAfterRefundReceived,0.0000,0x00000000,<NA>
18,0001437749-26-010731,FOREIGN,IncomeTaxPaidForeignAfterRefundReceived,0.0000,0x00000000,<NA>
19,0001437749-26-010731,FOREIGN,IncomeTaxPaidForeignAfterRefundReceived,240000.0000,0x6d05e900484fed80d7998640bf95a995,IncomeTaxAuthorityName=PeruTax;
20,0001437749-26-010731,FOREIGN,IncomeTaxPaidForeignAfterRefundReceived,60000.0000,0x91b5ab02843c18d53590bc2fd062d769,IncomeTaxAuthorityName=EcuadorTax;
21,0001437749-26-010731,FOREIGN,IncomeTaxPaidForeignAfterRefundReceived,379000.0000,0xbb538304d98e41613f61919a8bb10ef0,IncomeTaxAuthorityName=MexicanTaxAuthority;
22,0001437749-26-010731,STATE_LOCAL,IncomeTaxPaidStateAndLocalAfterRefundReceived,60000.0000,0x00000000,<NA>
16,0001437749-26-010731,TOTAL,IncomeTaxesPaidNet,739000.0000,0x00000000,<NA>


In [39]:
# A1.5.1A-6 — Step 2: Geographic reconciliation diagnostic

results = []

for company, accession in geo_accessions.items():

    # Select one company's facts
    df = geo_review.loc[
        geo_review["adsh"].eq(accession)
    ].copy()

    df["amount"] = pd.to_numeric(df["value"])

    # Original four facts: no geographic dimension
    core = df.loc[
        df["dimh"].eq("0x00000000")
    ].set_index("tax_type")["amount"]

    assert set(core.index) == {
        "TOTAL", "FEDERAL", "STATE_LOCAL", "FOREIGN"
    }
    assert core.index.is_unique

    strict_difference = (
        core["TOTAL"]
        - core["FEDERAL"]
        - core["STATE_LOCAL"]
        - core["FOREIGN"]
    )

    # Additional geographic facts
    geo = df.loc[
        df["dimh"].ne("0x00000000")
        & df["tax_type"].isin(["STATE_LOCAL", "FOREIGN"])
    ]

    geographic_sum = geo["amount"].sum()

    candidate_extended_difference = (
        strict_difference - geographic_sum
    )

    results.append({
        "company": company,
        "strict_difference": strict_difference,
        "n_geo_facts": len(geo),
        "geographic_sum": geographic_sum,
        "candidate_extended_difference":
            candidate_extended_difference
    })

geo_diagnostic = pd.DataFrame(results)

display(geo_diagnostic)


,company,strict_difference,n_geo_facts,geographic_sum,candidate_extended_difference
0,ONE Group,726000.0,6,726000.0,0.0
1,Zumiez,1287000.0,2,1287000.0,0.0
2,American Shared,679000.0,3,679000.0,0.0


In [40]:
pilot = build_pilot_outputs(
    cash_tax_current, sub, tax_all, cal7, qa_groups, ROOT, OUTPUT)
filing_universe = pilot['filing_universe']
coverage_10k = pilot['coverage_10k']
partial_scopes = pilot['partial_scopes']
display(pilot['coverage_summary'])
display(pilot['exception_register'])
assert len(filing_universe) == 2340
assert coverage_10k['has_total'].sum() == 1159
assert len(complete) == 184 and complete['difference'].eq(0).sum() == 177


,measure,n_filings,denominator,coverage_rate
0,has_total,1159,2340,0.495299
1,has_federal,713,2340,0.304701
2,has_state_local,518,2340,0.221368
3,has_foreign,276,2340,0.117949
4,has_complete_four,184,2340,0.078632


,adsh,ddate,qtrs,uom,dimh,reported_total_usd,federal_usd,state_local_usd,foreign_usd,strict_component_sum,...,source_10k_url,evidence_note,final_classification,source_accession,source_dataset,company,rounding_status,evidence_source,unresolved_issue,accounting_geo_relationship_confirmed
0,0000091388-26-000014,20251231,4,USD,0x00000000,124000000.0,103000000.0,19000000.0,3000000.0,125000000.0,...,https://www.sec.gov/Archives/edgar/data/91388/...,10-K Note 13; decimals=-6; rounding-compatible...,ROUNDING_COMPATIBLE,0000091388-26-000014,2026_03_notes.zip,SMITHFIELD FOODS INC,ROUNDING_COMPATIBLE,https://www.sec.gov/Archives/edgar/data/91388/...,10-K Note 13; decimals=-6; rounding-compatible...,False
1,0001104659-26-032065,20251231,4,USD,0x00000000,1211000.0,131000.0,171000.0,183000.0,485000.0,...,,Six state-dimensional facts sum to 726000; add...,GEO_CANDIDATE_REVIEW,0001104659-26-032065,2026_03_notes.zip,"ONE GROUP HOSPITALITY, INC.",PRECISION_INCOMPATIBLE,,Six state-dimensional facts sum to 726000; add...,False
2,0001193125-26-104063,20260131,4,USD,0x00000000,5876000.0,4050000.0,748000.0,-209000.0,4589000.0,...,,Two authority-dimensional facts sum to 1287000...,GEO_CANDIDATE_REVIEW,0001193125-26-104063,2026_03_notes.zip,ZUMIEZ INC,PRECISION_INCOMPATIBLE,,Two authority-dimensional facts sum to 1287000...,False
3,0001193125-26-115851,20251231,4,USD,0x00000000,30767000.0,24453000.0,2870000.0,0.0,27323000.0,...,,Prior pilot called Other reconciled. Exact CAL...,OTHER_NUMERIC_MATCH_CAL_SIGN_REVIEW,0001193125-26-115851,2026_03_notes.zip,"AVEANNA HEALTHCARE HOLDINGS, INC.",PRECISION_INCOMPATIBLE,SEC ZIP cal.tsv + exact NUM tag/version/scope;...,Prior pilot called Other reconciled. Exact CAL...,False
4,0001437749-26-010731,20251231,4,USD,0x00000000,739000.0,0.0,60000.0,0.0,60000.0,...,,Three foreign authority facts sum to 679000; a...,GEO_CANDIDATE_REVIEW,0001437749-26-010731,2026_03_notes.zip,AMERICAN SHARED HOSPITAL SERVICES,PRECISION_INCOMPATIBLE,,Three foreign authority facts sum to 679000; a...,False
5,0001493152-26-008465,20251231,4,USD,0x00000000,76110000.0,19911000.0,3246000.0,52901000.0,76058000.0,...,,Prior pilot called Other reconciled. Exact CAL...,OTHER_NUMERIC_MATCH_CAL_SIGN_REVIEW,0001493152-26-008465,2026_03_notes.zip,TECNOGLASS INC.,PRECISION_INCOMPATIBLE,SEC ZIP cal.tsv + exact NUM tag/version/scope;...,Prior pilot called Other reconciled. Exact CAL...,False
6,0001836981-26-000018,20251231,4,USD,0x00000000,0.0,0.0,-28000.0,0.0,-28000.0,...,https://www.sec.gov/Archives/edgar/data/183698...,10-K 2025 fixed-zero IncomeTaxesPaidNet fact f...,TOTAL_SEMANTIC_REVIEW,0001836981-26-000018,2026_03_notes.zip,"BIGBEAR.AI HOLDINGS, INC.",PRECISION_INCOMPATIBLE,https://www.sec.gov/Archives/edgar/data/183698...,10-K 2025 fixed-zero IncomeTaxesPaidNet fact f...,False


## 8. A1.6 — Filing-level Total eligibility
Audit every original 10-K, with a mutually exclusive final status and all subsidiary
diagnostics retained. Any-period tag presence and numeric facts are wider populations
than the original raw-coverage definition. Multiple versions/scopes are not summed.
Explicit zero is retained. A mechanically passing observation is only
`ELIGIBLE_PROVISIONAL`: universal filing-by-filing semantic verification is incomplete.
Original 10-K/A relationships are logged separately and never silently substituted.


In [41]:
total_eligibility, total_summary, count_bridge = audit_total_eligibility(
    sub, tax_all, tax_selected, qa_groups, ROOT, OUTPUT)
display(count_bridge)
display(total_summary)
assert total_eligibility['adsh'].is_unique
assert len(total_eligibility) == len(filing_universe)
assert total_eligibility['eligibility_status'].notna().all()
assert total_eligibility.loc[~total_eligibility['analysis_eligible'], 'exclusion_reason'].ne('').all()
bb = total_eligibility.set_index('adsh').loc['0001836981-26-000018']
assert bb['reported_total_usd'] == 0 and bb['reported_zero']
assert bb['semantic_status'] == 'LEGACY_ROW_SCOPE_AMBIGUOUS'
assert not bb['analysis_eligible'] and pd.isna(bb['analytical_total_usd'])


,stage,n_filings,reduction_from_previous,definition
0,01_ORIGINAL_10K,2340,0,"Original 10-K, not 10-K/A"
1,02_TOTAL_TAG_ANY_PERIOD,1294,1046,Standard Total tag present in any period/scope
2,03_NUMERIC_TOTAL_ANY_PERIOD,1211,83,At least one nonmissing numeric Total
3,04_CURRENT_ANNUAL_NUMERIC,1172,39,"fp FY, qtrs 4, standardized ddate equals period"
4,05_DIMENSIONLESS_USD_CURRENT,1159,13,"Original 1,159 coverage definition"
5,06_UNIQUE_DUPLICATE_REVIEWED,1158,1,Unique target fact; duplicate audit; prior pre...
6,07_SEMANTIC_EXCLUSIONS_APPLIED,1157,1,Exclude known legacy-row ambiguity; no univers...
7,08_ELIGIBLE_PROVISIONAL,1157,0,Mechanical checks pass; semantic validity prov...


,eligibility_status,n_filings,n_reported_zero,share_of_original_10k
0,ELIGIBLE_PROVISIONAL,1157,123,0.494444
1,EXCLUDED_NO_TOTAL_TAG,1046,0,0.447009
2,EXCLUDED_SEMANTIC_AMBIGUITY,1,1,0.000427
3,EXCLUDED_TOTAL_VALUE_MISSING,83,0,0.035470
4,REVIEW_DIMENSIONED_TOTAL_ONLY,13,0,0.005556
5,REVIEW_NO_CURRENT_ANNUAL_NUMERIC,39,0,0.016667
6,REVIEW_UNVERIFIED_PRECISION_SOURCE,1,1,0.000427


## 9. Acceptance checks and reproducibility record
Recomputed Strict facts and coverage are checked against the frozen outputs.
The reference is used only for validation, never to calculate reported amounts.
Both raw inputs and development files remain unchanged. Outputs describe Stage 1
data construction, not an economically validated final sample or paper replication.


In [42]:
reference = pd.read_csv(ROOT/'evidence/pilot_reconciliation_reference.csv', dtype='string')
reproduced = pd.read_csv(OUTPUT/'cash_tax_reconciliation_184.csv', dtype='string')
keys = ['adsh','ddate','qtrs','uom','dimh']
numeric = ['reported_total_usd','federal_usd','state_local_usd','foreign_usd','strict_component_sum','strict_difference']
a = reference.set_index(keys).sort_index()
b = reproduced.set_index(keys).sort_index()
assert a.index.equals(b.index)
for col in numeric:
    pd.testing.assert_series_equal(pd.to_numeric(a[col]), pd.to_numeric(b[col]), check_dtype=False)
assert len(pilot['exception_register']) == 7
assert pilot['exception_register'].set_index('adsh').loc['0000091388-26-000014','rounding_status'] == 'ROUNDING_COMPATIBLE'
assert not reproduced['rounding_confirmed'].eq('True').any()
assert hashlib.file_digest(notes_path.open('rb'), 'sha256').hexdigest() == raw_sha256
checks = {'filing_universe':len(filing_universe),
          'original_numeric_total_coverage':int(coverage_10k['has_total'].sum()),
          'strict_scopes':len(complete),'strict_exact':int(complete['difference'].eq(0).sum()),
          'exceptions':len(pilot['exception_register']),
          'provisional_eligible':int(total_eligibility['analysis_eligible'].sum()),
          'reported_zero_selected':int(total_eligibility['reported_zero'].sum()),
          'original_strict_parity':True,'raw_sha256':raw_sha256}
(OUTPUT/'acceptance_checks.json').write_text(json.dumps(checks,indent=2),encoding='utf-8')
print(json.dumps(checks,indent=2))


{
  "filing_universe": 2340,
  "original_numeric_total_coverage": 1159,
  "strict_scopes": 184,
  "strict_exact": 177,
  "exceptions": 7,
  "provisional_eligible": 1157,
  "reported_zero_selected": 125,
  "original_strict_parity": true,
  "raw_sha256": "d1c1711e6cee2f197fb91879521da178329d1c1ed369f9086aa501aff45f83d4"
}
